# GTEx model building with PLIER

💡 **Environment:** `clamp-analyses`  

This notebook builds latent variable models from GTEx v8 RNA‑seq TPM data using PLIER. It automates downloading and preprocessing the GTEx matrix, creates a Filebacked Big Matrix (FBM), computes an SVD to estimate the model dimension, prepares pathway priors, runs PLIER, and saves model outputs (B, Z, summaries) and intermediate files. Configuration and paths are controlled via `config.R`.

## Load libraries

In [1]:
# Create a timestamp to track the start of the analysis
start_time <- Sys.time()
cat("GTEx PLIER analysis started at:", format(start_time), "\n")

GTEx PLIER analysis started at: 2026-03-04 14:28:00 


In [2]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(PLIER)
library(CLAMP)

source(here("config.R"))

set.seed(config$GTEx$RANDOM_SVD_SEED)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses

Loading required package: RColorBrewer

Loading required package: gplots


---------------------
gplots 3.3.0 loaded:
  * Use citation('gplots') for citation info.
  * Homepage: https://talgalili.github.io/gplots/
  * Report issues: https://github.com/talgalili/gplots/issues
  * Ask questions: https://stackoverflow.com/questions/tagged/gplots
  * Suppress this message with: suppressPackageStartupMessages(library(gplots))
---------------------



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess


Loading required package: pheatmap

Loadin

## Output directory

In [3]:
output_data_dir <- config$GTEx$OUTPUT_DIR
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

output_data_dir

[1] "/home/msubirana/Documents/pivlab/clamp-analyses/output/gtex"

## Input

In [4]:
gtex_fbm_filt <- readRDS(file.path(output_data_dir, "gtex_fbm_filt.rds"))
gtex_svdRes <- readRDS(file.path(output_data_dir, "gtex_svdRes.rds"))
CLAMP_K_gtex <- readRDS(file.path(output_data_dir, "CLAMP_K_gtex.rds"))
gtex_genes <- readRDS(file.path(output_data_dir, "gtex_genes.rds"))
samples <- readRDS(file.path(output_data_dir, "gtex_samples.rds"))

# Settings

In [5]:
block_size <- config$GENERAL$CHUNK_SIZE
N_CORES    <- config$GTEx$N_CORES

## Prepare pathway priors

In [6]:
data_path <- here::here('data/archs4')

c2_gmt <- CLAMP:::read_gmt(file.path(data_path, "c2.cp.v2026.1.Hs.symbols.gmt"))
names(c2_gmt) <- paste0("C2CP_", names(c2_gmt))

gtex_pathMat <- gmtListToSparseMat(list(C2CP = c2_gmt))
gtex_matched  <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)
gtex_chatObj <- getChat(gtex_matched)

There are 11250 genes in the intersection between data and prior

Removing 980 pathways

Inverting...

done



# PLIER

Run PLIER with the same inputs

In [7]:
gtex_plier = PLIER::PLIER(
    gtex_fbm_filt[], 
    as.matrix(gtex_matched), 
    svdres = gtex_svdRes, 
    Chat = as.matrix(gtex_chatObj), 
    doCrossval = TRUE, 
    k = CLAMP_K_gtex
  )

Removing 0 pathways with too few genes



[1] 135.8334
[1] "L2 is set to 135.833443715207"
[1] "L1 is set to 67.9167218576034"


errorY (SVD based:best possible) = 0.3614

New L3 is 0.000203468369010644

New L3 is 0.000139841628594101

New L3 is 0.000158461325115751

New L3 is 0.000139841628594101

New L3 is 0.000139841628594101

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.000139841628594101

New L3 is 0.000139841628594101

New L3 is 0.000139841628594101

New L3 is 0.000158461325115751

New L3 is 0.000139841628594101

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

converged at  iteration 255 Bdiff is not decreasing

There are 122  LVs with AUC>0.70



In [8]:
colnames(gtex_plier$Z) <- paste0('LV', seq_len(ncol(gtex_plier$Z)))

In [9]:
head(gtex_plier$Z)
dim(gtex_plier$Z)

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,⋯,LV403,LV404,LV405,LV406,LV407,LV408,LV409,LV410,LV411,LV412
WASH7P,0.06691270,0.03804431,0.0000000,0.00000000,0.07012075,0.02591740,0.0000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.0000000,0.101966,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
RP11-34P13.15,0.00000000,0.04939260,0.0000000,0.63851291,0.00000000,0.30439509,0.0000000,0.09700285,0.00000000,0.00000000,⋯,0.03944939,0.0000000,0.000000,0.03161584,0.00000000,0.00000000,0.00000000,0.00000000,0.06512196,0.06544059
RP11-34P13.16,0.10056369,0.03612933,0.0000000,0.69879373,0.00000000,0.29401440,0.0000000,0.11839440,0.00000000,0.00000000,⋯,0.01987714,0.0000000,0.000000,0.02308379,0.00000000,0.00000000,0.00000000,0.00000000,0.05513001,0.05418699
RP11-34P13.18,0.13666388,0.00000000,0.0144381,0.00000000,0.08266562,0.17242953,0.0000000,0.00000000,0.04925944,0.00000000,⋯,0.00000000,0.0000000,0.000000,0.00000000,0.00000000,0.05800589,0.03088884,0.00000000,0.00000000,0.01204653
AP006222.2,0.07779624,0.00000000,0.0000000,0.00000000,0.07140100,0.05457484,0.0000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.1045995,0.000000,0.06919823,0.03811857,0.00000000,0.00000000,0.00000000,0.00000000,0.18709749
MTND1P23,0.13927177,0.00000000,0.0000000,0.04749296,0.10191688,0.02879276,0.3901867,0.00000000,0.00000000,0.03524977,⋯,0.06685109,0.0000000,0.000000,0.15617769,0.03926147,0.02740168,0.00000000,0.05037789,0.00000000,0.00000000


[1] 21613   412

In [10]:
gtex_plier$summary <- gtex_plier$summary %>%
    dplyr::rename(LV = `LV index`)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

In [11]:
head(gtex_plier$summary)

,pathway,LV,AUC,p-value,FDR
,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,C2CP_KEGG_CELL_CYCLE,LV1,0.8675507,1.785724e-10,2.912013e-09
2,C2CP_KEGG_CYTOSOLIC_DNA_SENSING_PATHWAY,LV1,0.6851278,4.391834e-02,7.620045e-02
3,C2CP_KEGG_REGULATION_OF_ACTIN_CYTOSKELETON,LV1,0.6779074,5.488591e-05,2.554857e-04
4,C2CP_REACTOME_AEROBIC_RESPIRATION_AND_RESPIRATORY_ELECTRON_TRANSPORT,LV1,0.7066639,6.980428e-07,5.476782e-06
5,C2CP_REACTOME_ANTIGEN_PROCESSING_UBIQUITINATION_PROTEASOME_DEGRADATION,LV1,0.7377026,3.018516e-10,4.796940e-09
6,C2CP_REACTOME_APOPTOTIC_EXECUTION_PHASE,LV1,0.7295797,8.196321e-03,1.859075e-02


In [12]:
head(gtex_plier$B)
dim(gtex_plier$B)

"1,C2CP_REACTOME_METABOLISM_OF_RNA",0.36996369,-0.003232835,0.39565715,0.044528742,-0.369253733,-0.063628216,0.18827702,0.36435454,0.17250529,0.41668064,⋯,-0.112885092,0.14312931,0.12574626,0.052010572,0.456378945,0.4800822909,0.692138523,0.46866065,0.22155684,0.22469022
LV 2,0.04937153,0.473040190,0.02589191,0.256100405,0.132724932,-0.004531794,-0.09174624,-0.06771513,0.13520579,0.06543548,⋯,0.058580464,-0.03801269,-0.19815338,0.007304977,0.153142950,-0.3523653625,-0.104140131,0.06090273,0.12046946,0.13482956
"3,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_PANCREATIC_ACINAR_CELLS",-0.05379928,0.025935853,0.06762199,0.005497956,0.007138144,-0.005614075,0.01256331,-0.06569057,-0.09913638,-0.04421470,⋯,-0.008469078,-0.03899084,-0.07832742,-0.061279952,0.005927443,0.0004120762,-0.009769566,0.02892308,-0.02964380,-0.07062038
"4,C2CP_REACTOME_NEUTROPHIL_DEGRANULATION",-0.11882533,0.018203790,-0.04112356,-0.147879857,0.036174434,-0.184118811,-0.09180944,-0.12225625,-0.10630487,-0.11678403,⋯,0.013902132,-0.16031285,-0.06675617,-0.042808470,-0.079989409,-0.0768905497,-0.023591391,0.05764444,0.01582593,-0.01862774
"5,C2CP_WP_TRANSLATION_FACTORS",0.02963275,1.279887130,-0.03857454,-0.050858223,-0.008105372,-0.130136831,-0.08763174,-0.14559746,-0.02555721,-0.03227134,⋯,0.144260099,-0.05107629,-0.09565038,-0.058735228,0.140278780,-0.3043679322,-0.078670978,0.04500140,2.03341888,0.12425954
"6,C2CP_WP_CILIOPATHIES",-0.02820884,0.010497850,-0.03215512,-0.047489885,-0.084866247,-0.110568574,-0.07709691,-0.13338378,-0.05363429,-0.14810945,⋯,-0.081609865,-0.09903088,-0.02224543,-0.074019301,-0.024783923,-0.1852713912,-0.160723361,-0.05068422,-0.08510815,-0.04976613


[1]   412 17382

In [13]:
colnames(gtex_plier$B) <- samples

In [14]:
saveRDS(gtex_plier, file = file.path(output_data_dir, "PLIER_BP.rds"))

In [15]:
model_dir <- file.path(output_data_dir, "PLIER_BP")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- gtex_plier$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- gtex_plier$Z
write.csv(Z, file.path(model_dir, "Z.csv"))

summary <- gtex_plier$summary
write.csv(summary, file.path(model_dir, "summary.csv"))